# Smart Association Advisor demo

This notebook demonstrates the player-facing advisor built on top of the saved clan-winner classifier.

The advisor evaluates candidate actions for every manager and prints clean manager cards. 

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd

from src.common.features import prepare_members
from src.advisor.rendering import render_matchup_advice

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 42

DATA_DIR = PROJECT_ROOT / 'data' / 'train'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'

MATCHES_FILE = DATA_DIR / 'clan_matches_training.csv'
MEMBERS_FILE = DATA_DIR / 'member_stats_training.csv'
MODEL_FILE = ARTIFACT_DIR / 'clan_winner_classifier_artifact.joblib'

In [2]:
members = pd.read_csv(MEMBERS_FILE)
matches = pd.read_csv(MATCHES_FILE)

artifact = joblib.load(MODEL_FILE)
model = artifact['model']
feature_cols = artifact['feature_cols']
VALUE_COLS = artifact['value_cols']

members = prepare_members(members, value_cols=VALUE_COLS)

print('members:', members.shape)
print('matches:', matches.shape)
print('loaded features:', len(feature_cols))
print('model:', type(model).__name__)

members: (291456, 15)
matches: (24288, 5)
loaded features: 72
model: XGBClassifier


## Example matchup

The printed card below is the player-facing style.

In [3]:
example = matches.sample(1, random_state=RANDOM_STATE).iloc[0]
clan_a_id = example['clan_1_id']
clan_b_id = example['clan_2_id']

advice_text, manager_summary = render_matchup_advice(
    model=model,
    full_members=members,
    clan_a_id=clan_a_id,
    clan_b_id=clan_b_id,
    feature_cols=feature_cols,
    value_cols=VALUE_COLS,
)

print(advice_text)

############################################################################################
SMART ASSOCIATION ADVISOR
############################################################################################
Matchup : Clan clan_44207 vs Clan clan_43560
############################################################################################

CLAN clan_44207 — MANAGER ADVICE

1. Manager user_154960
  Priority : High priority
  Focus    : Prioritize training now
  Action   : Have this manager complete training now and keep bonuses high before matches.
  Why      : This manager's results count more for the Association, so missing training bonuses here can cost more.

2. Manager user_254405
  Priority : Moderate priority
  Focus    : Keep ready
  Action   : No urgent ping needed. Just keep normal training, bonuses, and lineup readiness in place.
  Why      : This manager's matches matter a lot, but their preparation already looks stable. Keep it that way.

3. Manager user_254263
  P

## More examples

These examples show the same rendering for several matchups.

In [4]:
examples = matches.sample(3, random_state=7)

for i, (_, row) in enumerate(examples.iterrows(), start=1):
    text, _ = render_matchup_advice(
        model=model,
        full_members=members,
        clan_a_id=row['clan_1_id'],
        clan_b_id=row['clan_2_id'],
        feature_cols=feature_cols,
        value_cols=VALUE_COLS,
    )

    print('\n' + '=' * 120)
    print(f'EXAMPLE {i}')
    print('=' * 120)
    print(text)
    print()


EXAMPLE 1
############################################################################################
SMART ASSOCIATION ADVISOR
############################################################################################
Matchup : Clan clan_20287 vs Clan clan_32339
############################################################################################

CLAN clan_20287 — MANAGER ADVICE

1. Manager user_240842
  Priority : High priority
  Focus    : Prioritize training now
  Action   : Have this manager complete training now and keep bonuses high before matches.
  Why      : This manager's results count more for the Association, so missing training bonuses here can cost more.

2. Manager user_29577
  Priority : High priority
  Focus    : Prioritize training now
  Action   : Have this manager complete training now and keep bonuses high before matches.
  Why      : This manager's results count more for the Association, so missing training bonuses here can cost more.

3. Manager user